# NeMo Agent Toolkit SDK: Installation & Getting Started

Welcome to the NeMo Agent Toolkit (NAT) Python SDK! This tutorial series will guide you through building AI agent workflows using Python, with the ability to export configurations for production deployment.

## Understanding NeMo Agent Toolkit's Paradigm

NeMo Agent Toolkit takes a fundamentally different approach from traditional Python agent frameworks. Instead of imperatively constructing agents in code, NAT uses a **configuration-driven paradigm** where workflows are defined declaratively in YAML files.

From the [Workflow Configuration documentation](../../../docs/source/workflows/workflow-configuration.md):

> *"NeMo Agent toolkit workflows are defined by a YAML configuration file, which specifies which entities (functions, LLMs, embedders, etc.) to use in the workflow, along with general configuration settings."*

### Why Configuration-Driven?

The YAML configuration file is a **flattened representation** of everything your agent can do—all functions, LLMs, embedders, retrievers, and memory components are declared upfront. This design provides significant advantages:

1. **Performance**: By loading all components asynchronously at startup, NAT can:
   - Resolve the dependency tree and initialize components in optimal order
   - Parallelize initialization of independent components
   - Establish connections (database, API, etc.) once and share them efficiently

2. **Reproducibility**: Configuration files are version-controllable, shareable, and provide a complete specification of your workflow

3. **Production-Ready**: The same configuration that works in development deploys directly to production via CLI

### Framework Compatibility

From the [Frameworks Overview](../../../docs/source/reference/frameworks-overview.md), NAT integrates with multiple agentic frameworks:

- **LangChain/LangGraph** - Full LLM, embedder, retriever, and tool calling support
- **LlamaIndex** - Full LLM, embedder, and tool calling support
- **CrewAI** - Multi-agent orchestration framework
- **Semantic Kernel** - Microsoft's SDK for LLM integration
- **Google ADK** - Google Agent Development Kit
- **Strands** - AWS AgentCore runtime for Bedrock
- **Agno** - Lightweight agent framework

### Where the SDK Fits In

While the configuration-driven approach is powerful, developers familiar with Python frameworks may find it easier to start by building workflows programmatically. The SDK bridges this gap:

- **Develop in Python**: Use familiar classes, IDE autocompletion, and type checking
- **Export to YAML**: Generate production-ready configuration files
- **Deploy with CLI**: Run workflows using `nat run`, `nat serve`, or integrate into CI/CD

The SDK generates the exact same YAML configurations you would write manually, ensuring a seamless transition from development to production.

## SDK vs Configuration Approach

| Aspect | SDK (Python) | YAML Configuration |
|--------|--------------|-------------------|
| **Best For** | Rapid prototyping, notebooks, exploration | Production deployment, version control |
| **Learning Curve** | Familiar to Python developers | Requires understanding NAT's structure |
| **IDE Support** | Full autocompletion, type hints | Schema validation, syntax highlighting |
| **Workflow** | Build → Export → Deploy | Edit → Run |

**Recommended Workflow**: Start with the SDK to prototype and iterate quickly, then export to YAML for production deployment.


## Installation

### Option 1: Install from PyPI (Recommended for Users)

```bash
# Core package
uv pip install nvidia-nemo-agent-toolkit

# With optional plugins
uv pip install nvidia-nemo-agent-toolkit[mcp]      # MCP support
uv pip install nvidia-nemo-agent-toolkit[a2a]      # A2A protocol support
uv pip install nvidia-nemo-agent-toolkit[all]      # All plugins
```

### Option 2: Install from Source (Recommended for Development)

```bash
# Clone the repository
git clone https://github.com/NVIDIA/nemo-agent-toolkit.git
cd nemo-agent-toolkit

# Create virtual environment with uv
uv venv .venv --python 3.12
source .venv/bin/activate

# Install in development mode
uv pip install -e ".[dev]"

# Install plugins as needed
uv pip install -e packages/nvidia_nat_mcp
uv pip install -e packages/nvidia_nat_a2a
```

### Verify Installation

```bash
# Check CLI is available
nat --version

# Check available commands
nat --help
```


## Environment Setup

The SDK requires API keys for the LLM providers you plan to use. Set these as environment variables:


In [1]:
import getpass
import os

from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Check for NVIDIA API key (required for NIM models)
# Get yours at: https://build.nvidia.com/
nvidia_api_key = os.environ.get("NVIDIA_API_KEY")

if nvidia_api_key:
    print("✅ NVIDIA_API_KEY loaded")
else:
    nvidia_api_key = getpass.getpass("Enter your NVIDIA_API_KEY (get one at https://build.nvidia.com/): ")
    if nvidia_api_key:
        os.environ["NVIDIA_API_KEY"] = nvidia_api_key
        print("✅ NVIDIA_API_KEY set")
    else:
        print("⚠️ NVIDIA_API_KEY not set - NIM examples will fail")


✅ NVIDIA_API_KEY loaded


## The Function-Centric Model

According to the [Functions documentation](../../../docs/source/workflows/functions/index.md):

> *"Functions (tools) are the main building blocks of NeMo Agent toolkit and define the logic of your workflow."*

Functions are type-safe, asynchronous operations with support for both single and streaming outputs. They provide:
- **Type validation** via Pydantic schemas
- **Composability** through unified interfaces
- **Asynchronous operation** for better performance and scalability

### Component Types

NeMo Agent Toolkit provides several component types, all of which work together through the function abstraction:

| Component | Documentation | Description |
|-----------|--------------|-------------|
| **Functions** | [Functions](../../../docs/source/workflows/functions/index.md) | Individual tools with defined input/output schemas |
| **Function Groups** | [Function Groups](../../../docs/source/workflows/function-groups.md) | Package related functions to share configuration and resources |
| **Agents** | [About Workflows](../../../docs/source/workflows/about.md) | Systems that use LLMs to reason and execute functions |
| **LLMs** | [LLMs](../../../docs/source/workflows/llms/index.md) | Language model providers (NIM, OpenAI, AWS Bedrock, Azure, LiteLLM) |
| **Embedders** | [Embedders](../../../docs/source/workflows/embedders.md) | Embedding providers (NIM, OpenAI, Azure OpenAI) |
| **Retrievers** | [Retrievers](../../../docs/source/workflows/retrievers.md) | Vector database integration (NeMo Retriever, Milvus) |
| **Memory** | [Memory](../../../docs/source/store-and-retrieve/memory.md) | Long-term storage (Mem0, Redis, Zep) |
| **MCP Client** | [MCP](../../../docs/source/workflows/mcp/index.md) | Connect to tools via Model Context Protocol |
| **A2A Client** | [A2A](../../../docs/source/workflows/a2a/index.md) | Agent-to-agent communication and delegation |

### Supported Agents

From the [About Workflows](../../../docs/source/workflows/about.md) documentation:

| Agent | Description |
|-------|-------------|
| **ReAct Agent** | Reasoning and acting with explicit thought process |
| **Tool Calling Agent** | Uses LLM's native function calling capabilities |
| **ReWOO Agent** | Planning-first approach for complex multi-step tasks |
| **Reasoning Agent** | Enhanced reasoning capabilities |
| **Router Agent** | Routes requests to appropriate sub-agents |
| **Sequential Executor** | Executes functions in sequence |
| **Responses API Agent** | OpenAI Responses API compatible agent |


## Quick Start: Hello World Agent

Let's create a minimal agent to verify everything is working:


In [2]:
import sys
from pathlib import Path

# Add src to path for development (not needed if installed via pip)
module_path = Path("../../../src").resolve()
if str(module_path) not in sys.path:
    sys.path.insert(0, str(module_path))


In [3]:
from nat.agent.sdk import NatReActAgent
from nat.llm.sdk import NimLLM
from nat.tool.sdk import CurrentTimeTool
from nat.utils.sdk.nat_workflow import NatWorkflow

# Step 1: Create an LLM
llm = NimLLM(
    model_name="meta/llama-3.3-70b-instruct",
    temperature=0.0,
    name="nim_llm",
)

# Step 2: Create a tool (using built-in CurrentTimeTool)
time_tool = CurrentTimeTool(name="current_time")

# Step 3: Create an agent with the tool
agent = NatReActAgent(
    tools=[time_tool],
    llm=llm,
    verbose=True,
)

# Step 4: Wrap in a workflow
workflow = NatWorkflow(entrypoint=agent)

print("✅ Workflow created successfully!")


/Users/spastoriza/Documents/Programming/public/nat-fork/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


✅ Workflow created successfully!


## Save Configuration

One of the SDK's key features is the ability to export your Python workflow to a YAML configuration file:


In [4]:
# Create output directory
config_dir = Path("./configs")
config_dir.mkdir(parents=True, exist_ok=True)

# Save the workflow configuration
config_path = config_dir / "hello_world.yaml"
workflow.save_to_config_file(config_path)

print(f"📄 Configuration saved to: {config_path}")
print("\n" + "=" * 60)
print("GENERATED CONFIGURATION:")
print("=" * 60 + "\n")

with open(config_path) as f:
    print(f.read())


📄 Configuration saved to: configs/hello_world.yaml

GENERATED CONFIGURATION:

functions:
  current_time:
    _type: current_datetime

llms:
  nim_llm:
    _type: nim
    model: meta/llama-3.3-70b-instruct
    temperature: 0.0

workflow:
  _type: react_agent
  llm_name: nim_llm
  verbose: true
  tool_names:
  - current_time



## Run via CLI

Now you can run the workflow using the NAT CLI:

```bash
# Run the workflow with an input
nat run --config_file configs/hello_world.yaml --input "What time is it right now?"

# Serve as an API
nat serve --config_file configs/hello_world.yaml --port 8000
```

## Run in Python

You can also run the workflow directly in Python:


In [5]:
# Run the workflow (uncomment to execute)
result = await workflow.prompt("What time is it right now?")
print(result)


The current time is 2025-12-30 00:13:03 +0000.


## Tutorial Series Overview

This tutorial series covers:

| Tutorial | Topic | Description |
|----------|-------|-------------|
| **01** | Installation & Getting Started | This tutorial |
| **02** | Your First Agent | Building a complete calculator agent |
| **03** | Agent Types | ReAct, Tool Calling, ReWOO agents |
| **04** | Functions & Tools | Custom functions, MCP, A2A |
| **05** | LLM Providers | NIM, OpenAI, and configuration |
| **06** | Memory | Adding memory to agents |
| **07** | Retrievers | RAG with vector databases |
| **08** | Configuration Guide | YAML structure and options |
| **09** | Evaluation | Testing workflow performance |
| **10** | Profiling | Measuring latency and costs |
| **11** | Optimization | Improving prompts and parameters |
| **12** | Observability | Tracing and monitoring |

## Next Steps

Continue to **[02_your_first_agent.ipynb](./02_your_first_agent.ipynb)** to build a complete calculator agent!
